In [1]:
import pyfolio as pf

/opt/anaconda3/lib/python3.9/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(


In [3]:
import tushare as ts 

In [7]:
import pymc3 as pm

In [4]:
pro = ts.pro_api('fe8102bf83f5f83f6608aa46fa5e985c534c227786236a1192e5fd55')

df = pro.daily(ts_code='300750.SZ', start_date='20140701', end_date='20211231')


In [5]:
#stock_rets = pf.utils.get_symbol_rets('FB')
df.head()

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,300750.SZ,20211231,598.3,605.99,582.00,588.00,590.00,-2.00,-0.3390,93711.17,5540408.920
1,300750.SZ,20211230,577.0,596.29,575.57,590.00,575.55,14.45,2.5106,124209.46,7323088.709
2,300750.SZ,20211229,585.0,585.99,569.78,575.55,585.00,-9.45,-1.6154,99639.63,5731323.286
3,300750.SZ,20211228,578.0,587.28,569.91,585.00,578.60,6.40,1.1061,96595.74,5589310.230
4,300750.SZ,20211227,574.2,587.99,573.50,578.60,576.80,1.80,0.3121,101835.43,5915820.121


# Teear_Sheets

Tear sheets
Collection of tables and plots.

Various tear sheets based on:

returns
positions
transactions
periods of market stress
Bayesian analyses

In [9]:
df.describe()

,open,high,low,close,pre_close,change,pct_chg,vol,amount
count,868.000000,868.000000,868.000000,868.000000,868.000000,868.000000,868.000000,8.680000e+02,8.680000e+02
mean,218.425588,223.338548,213.457189,218.561751,217.912604,0.649147,0.426092,1.539581e+05,3.350073e+06
std,179.454662,183.241121,175.006533,179.329525,179.009224,9.046432,3.604514,1.011903e+05,2.914081e+06
min,30.170000,36.200000,30.170000,36.200000,25.140000,-45.270000,-10.003300,2.657900e+02,1.058375e+03
25%,76.000000,77.382500,75.000000,75.975000,75.900000,-2.090000,-1.593450,9.220190e+04,7.949777e+05
50%,137.165000,140.385000,135.005000,137.285000,136.310000,0.120000,0.130800,1.304067e+05,2.515970e+06
75%,334.250000,341.902500,325.170000,334.330000,333.210000,3.012500,1.893750,1.866864e+05,5.166973e+06
max,688.960000,692.000000,673.330000,688.000000,688.000000,52.990000,43.990000,1.281706e+06,1.450253e+07


In [16]:
from datetime import timedelta
from datetime import date
from datetime import datetime

In [20]:
df.pct_chg.asdatetime()

AttributeError: 'Series' object has no attribute 'asdatetime'

In [14]:
df.index(df.trade_date)

TypeError: 'RangeIndex' object is not callable

In [8]:
pf.create_returns_tear_sheet(df.pct_chg)

/opt/anaconda3/lib/python3.9/site-packages/empyrical/stats.py:1494: RuntimeWarning: invalid value encountered in log1p
  cum_log_returns = np.log1p(returns).cumsum()


AttributeError: 'int' object has no attribute 'strftime'

Zipline + pyfolio
Open-source backtester by Quantopian Inc.
Powers Quantopian.com
Various models for transaction costs and slippage.

In [ ]:
# Zipline trading algorithm
# Taken from zipline.examples.olmar
zipline_logging = logbook.NestedSetup([
    logbook.NullHandler(level=logbook.DEBUG),
    logbook.StreamHandler(sys.stdout, level=logbook.INFO),
    logbook.StreamHandler(sys.stderr, level=logbook.ERROR),
])
zipline_logging.push_application()

STOCKS = ['AMD', 'CERN', 'COST', 'DELL', 'GPS', 'INTC', 'MMM']


In [ ]:
# On-Line Portfolio Moving Average Reversion

# More info can be found in the corresponding paper:
# http://icml.cc/2012/papers/168.pdf
def initialize(algo, eps=1, window_length=5):
    algo.stocks = STOCKS
    algo.sids = [algo.symbol(symbol) for symbol in algo.stocks]
    algo.m = len(algo.stocks)
    algo.price = {}
    algo.b_t = np.ones(algo.m) / algo.m
    algo.last_desired_port = np.ones(algo.m) / algo.m
    algo.eps = eps
    algo.init = True
    algo.days = 0
    algo.window_length = window_length
    algo.add_transform('mavg', 5)

    algo.set_commission(commission.PerShare(cost=0))


In [ ]:
def handle_data(algo, data):
    algo.days += 1
    if algo.days < algo.window_length:
        return

    if algo.init:
        rebalance_portfolio(algo, data, algo.b_t)
        algo.init = False
        return

    m = algo.m

    x_tilde = np.zeros(m)
    b = np.zeros(m)

    # find relative moving average price for each asset
    for i, sid in enumerate(algo.sids):
        price = data[sid].price
        # Relative mean deviation
        x_tilde[i] = data[sid].mavg(algo.window_length) / price

    ###########################
    # Inside of OLMAR (algo 2)
    x_bar = x_tilde.mean()

    # market relative deviation
    mark_rel_dev = x_tilde - x_bar

    # Expected return with current portfolio
    exp_return = np.dot(algo.b_t, x_tilde)
    weight = algo.eps - exp_return
    variability = (np.linalg.norm(mark_rel_dev)) ** 2

    # test for divide-by-zero case
    if variability == 0.0:
        step_size = 0
    else:
        step_size = max(0, weight / variability)

    b = algo.b_t + step_size * mark_rel_dev
    b_norm = simplex_projection(b)
    np.testing.assert_almost_equal(b_norm.sum(), 1)

    rebalance_portfolio(algo, data, b_norm)

    # update portfolio
    algo.b_t = b_norm

In [ ]:
def rebalance_portfolio(algo, data, desired_port):
    # rebalance portfolio
    desired_amount = np.zeros_like(desired_port)
    current_amount = np.zeros_like(desired_port)
    prices = np.zeros_like(desired_port)

    if algo.init:
        positions_value = algo.portfolio.starting_cash
    else:
        positions_value = algo.portfolio.positions_value + \
            algo.portfolio.cash

    for i, sid in enumerate(algo.sids):
        current_amount[i] = algo.portfolio.positions[sid].amount
        prices[i] = data[sid].price

    desired_amount = np.round(desired_port * positions_value / prices)

    algo.last_desired_port = desired_port
    diff_amount = desired_amount - current_amount

    for i, sid in enumerate(algo.sids):
        algo.order(sid, diff_amount[i])


def simplex_projection(v, b=1):
    """Projection vectors to the simplex domain

    Implemented according to the paper: Efficient projections onto the
    l1-ball for learning in high dimensions, John Duchi, et al. ICML 2008.
    Implementation Time: 2011 June 17 by Bin@libin AT pmail.ntu.edu.sg
    Optimization Problem: min_{w}\| w - v \|_{2}^{2}
    s.t. sum_{i=1}^{m}=z, w_{i}\geq 0

    Input: A vector v \in R^{m}, and a scalar z > 0 (default=1)
    Output: Projection vector w

    :Example:
    >>> proj = simplex_projection([.4 ,.3, -.4, .5])
    >>> print(proj)
    array([ 0.33333333, 0.23333333, 0. , 0.43333333])
    >>> print(proj.sum())
    1.0

    Original matlab implementation: John Duchi (jduchi@cs.berkeley.edu)
    Python-port: Copyright 2013 by Thomas Wiecki (thomas.wiecki@gmail.com).
    """

    v = np.asarray(v)
    p = len(v)

    # Sort v into u in descending order
    v = (v > 0) * v
    u = np.sort(v)[::-1]
    sv = np.cumsum(u)

    rho = np.where(u > (sv - b) / np.arange(1, p + 1))[0][-1]
    theta = np.max([0, (sv[rho] - b) / (rho + 1)])
    w = (v - theta)
    w[w < 0] = 0
    return w

start = datetime(2004, 1, 1, 0, 0, 0, 0, pytz.utc)
end = datetime(2010, 1, 1, 0, 0, 0, 0, pytz.utc)
data = load_from_yahoo(stocks=STOCKS, indexes={}, start=start, end=end)
data = data.dropna()
olmar = TradingAlgorithm(handle_data=handle_data,
                         initialize=initialize,
                         identifiers=STOCKS)
backtest = olmar.run(data)

Converting data from zipline to pyfolio

In [ ]:
returns, positions, transactions = \
    pf.utils.extract_rets_pos_txn_from_zipline(backtest)

In [ ]:
returns.tail()

In [ ]:
positions.tail()

In [ ]:
transactions.tail()

In [ ]:
oos_date = '2009-10-21'

pf.create_full_tear_sheet(returns,
                          positions=positions,
                          transactions=transactions,
                          live_start_date=oos_date,
                          slippage=0.1,
                          sector_mappings=sector_map)

Tear sheets call individual plotting functions in pyfolio.plotting
Plotting functions call individual statistical functions in pyfolio.timeseries

In [ ]:
# Show overview of pyfolio.plotting submodule
[f for f in dir(pf.plotting) if 'plot_' in f]

In [ ]:
help(pf.plotting.plot_rolling_returns)

In [ ]:
oos_date = '2009-10-21'
pf.create_bayesian_tear_sheet(returns, live_start_date=oos_date)

In [ ]:
from zipline.api import order_target, record, symbol

In [ ]:

def initialize(context):
    context.i = 0
    context.asset = symbol('AAPL')


def handle_data(context, data):
    # Skip first 300 days to get full windows
    context.i += 1
    if context.i < 300:
        return

    # Compute averages
    # data.history() has to be called with the same params
    # from above and returns a pandas dataframe.
    short_mavg = data.history(context.asset, 'price', bar_count=100, frequency="1d").mean()
    long_mavg = data.history(context.asset, 'price', bar_count=300, frequency="1d").mean()

    # Trading logic
    if short_mavg > long_mavg:
        # order_target orders as many shares as needed to
        # achieve the desired number of shares.
        order_target(context.asset, 100)
    elif short_mavg < long_mavg:
        order_target(context.asset, 0)

    # Save values for later inspection
    record(AAPL=data.current(context.asset, 'price'),
           short_mavg=short_mavg,
           long_mavg=long_mavg)